# 非评分实验 - Weaviate API 入门


<div align="center">
  <img src="images/weaviate.png" alt="RAG Overview" width="10%">
</div>

欢迎来到 Weaviate API 的非评分实验！当你进入向量数据库的世界，会发现有多种方案可用于构建 RAG 系统。在本课程中，你将重点使用 [Weaviate API](https://weaviate.io/)。

本实验旨在帮助你通过动手实践快速上手 Weaviate，为后续作业做好准备。你将了解 Weaviate 的工作方式、核心能力以及如何高效使用其功能。完成本实验后，你将具备完成作业所需的工具与知识。

开始吧！


# 目录
- [ 1 - 介绍](#1)
  - [ 1.1 加载所需库](#1-1)
  - [ 1.2 - Weaviate Client](#1-2)
- [ 2 - 配置数据库](#2)
  - [ 2.1 创建 Collection](#2-1)
  - [ 2.2 配置向量化器](#2-2)
  - [ 2.3 Properties](#2-3)
  - [ 2.4 向 Collection 添加元素](#2-4)
- [ 3 - 在 collection 上查询](#3)
  - [ 3.1 过滤器](#3-1)
  - [ 3.2 语义搜索](#3-2)
  - [ 3.3 BM25 搜索](#3-3)
  - [ 3.4 混合搜索](#3-4)
  - [ 3.5 重排（Reranking）](#3-5)


---
<h4 style="color:black; font-weight:bold;">使用目录</h4>

JupyterLab 提供了便捷方式帮助你浏览本实验。目录位于左侧面板中的目录选项卡，如下图所示。

![TOC Location](images/toc.png)

---

<a id='1'></a>
## 1 - 介绍
---
<a id='1-1'></a>
### 1.1 加载所需库
运行下方单元以加载本实验所需库。

In [1]:
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import Filter
from typing import List
from tqdm import tqdm
import joblib
import weaviate
import re
from weaviate.util import generate_uuid5
from pprint import pprint
import os

In [ ]:
from utils import (
    suppress_subprocess_output, 
    generate_with_single_input, 
    print_object_properties,
    kill_processes_on_ports
)

# 在导入 flask_app 前清理相关端口进程
# 警告：若重复运行本单元，可能会结束当前内核
kill_processes_on_ports([5000, 8080, 8097, 50050, 50051])

import flask_app

Could not cache non-existence of file. Will ignore error and continue. Error: [Errno 30] Read-only file system: '.models/models--BAAI--bge-base-en-v1.5/.no_exist/a5beb1e3e68b9ab74eb54cfd186867f64f240e1a/adapter_config.json'


 * Serving Flask app 'flask_app'
 * Debug mode: off


<a id='1-2'></a>
### 1.2 - Weaviate Client

要在当前环境中使用 Weaviate API，首先需要启动 `client`。本课程中你将使用 *embedded client*，即在当前应用内嵌运行 Weaviate，而不是依赖独立部署的 Weaviate 实例。

当你首次启动 `Embedded Weaviate` 时，它会在 `persistence_data_path` 指定的位置创建数据存储文件。即使后续关闭 client、Embedded Weaviate 停止运行，数据仍会保存在该位置。

创建 Weaviate client 时，需要传入一个嵌入模型用于向量化。你可以使用不同模型，Weaviate 也提供了调用 `OpenAI` 等模型的模块。由于 OpenAI 是付费服务，本实验将使用本地模型进行向量化。

加载 OpenAI 模型的一个示例如下：


```Python
import weaviate
client = weaviate.connect_to_embedded(
    version="1.26.1",
    headers={
        "X-OpenAI-Api-Key": YOUR_OPENAI_API_KEY
    },
)
```

现在加载我们的 client！

函数 `suppress_subprocess_output()` 用于抑制 Weaviate 输出日志，避免实验界面过于杂乱。本实验不会深入这些日志；如果你好奇日志内容，可以移除该函数。它使用的参数包括：

- `persistence_data_path`：client 查找（并创建）向量数据库的路径。创建后会持久化存储，即使关闭 client 也不会删除。
- `environment_variables`：为本地嵌入服务正常运行所需的环境变量。

In [ ]:
with suppress_subprocess_output():
    client = weaviate.connect_to_embedded(
        persistence_data_path="./.collections", # collections 的保存与持久化路径（可自定义）
        environment_variables={
            "ENABLE_API_BASED_MODULES": "true", # 启用基于 API 的模块
            "ENABLE_MODULES": 'text2vec-transformers, reranker-transformers', # 本实验使用 transformer 模型
            "TRANSFORMERS_INFERENCE_API":"http://127.0.0.1:5000/", # Weaviate 用于向量化的推理端点
            "RERANKER_INFERENCE_API":"http://127.0.0.1:5000/" # Weaviate 用于重排的推理端点
        }
    )

定义好 `client` 后，最主要的操作就是：创建 collection、向其中添加元素，以及在其上执行查询。

<a id='2'></a>
## 2 - 配置数据库

---

本节将介绍本实验和作业中的核心对象：[collection](https://weaviate.io/developers/weaviate/manage-data/collections)。在 Weaviate 中，collection 是一组可被索引并用于检索的数据对象。回顾课程中的工作流：

<div align="center">
  <img src="images/workflow.png" alt="RAG Overview" width="60%">
</div>


<a id='2-1'></a>
### 2.1 创建 Collection

创建 collection 时需要设置一些参数。对本实验最关键的是：

- `name`：collection 名称。它是保存到内存中的名字，也是后续加载时需要使用的名字。
- `vectorizer_config`：向量化配置列表。你可以传入多个向量化器配置，即同一向量数据库可用不同嵌入模型向量化数据点。本实验只使用一个。

下面加载一个数据库来演示。

In [4]:
data = joblib.load("data.joblib")
print_object_properties(data[0])

place: Grand Canyon
state: Arizona
description: A stunning canyon with vast vistas and incredible geology.
best_season_to_visit: Spring, Fall
attractions: South Rim, Havasu Falls, Skywalk
budget: Moderate
user_ratings: 4.8
last_updated: 2023-10-01T00:00:00Z



该数据集包含可旅行地点及描述其位置的信息属性。这里的属性有 `place, state, description, best_season_to_visit, attractions, budget, user_ratings, last_updated`。创建 collection 时，你需要为字典中的每个键创建对应 property，并指定预期数据类型。

<a id='2-2'></a>
### 2.2 配置向量化器

如前所述，你将使用 `text2vec_transformers` 嵌入模型对数据进行向量化。配置时需要传入对应的 Configure 对象。配置向量化器时，可传入多个向量化器，从而在同一 collection 中为同一对象保存多种向量表示。你还可以选择由特定向量化器仅向量化特定属性。本课程只使用一个向量化器。并非所有属性都必须向量化，这取决于数据特点和你希望检索的信息。

这里我们使用以下属性进行向量化：

`place, state, description, best_season_to_visit, attractions, budget`

这些属性会先拼接再向量化。定义 property 时，你可以选择是否把属性名也纳入向量化文本。例如对 budget 字段，只有 "Moderate" 这个词可能信息不足，加入属性名会更有助于表达其语义。

In [ ]:
vectorizer_config = [Configure.NamedVectors.text2vec_transformers(
                name="vector", # 后续访问对象向量时使用的向量名称
                source_properties=['place', 'state', 'description', 'best_season_to_visit', 'attractions', 'budget'], # 参与向量化的属性，会在向量化前拼接
                vectorize_collection_name = False, # 是否将 collection 名称也纳入向量化
                                                   # 若为 True，会拼接到待向量化文本开头
                inference_url="http://127.0.0.1:5000", # 使用 API 向量化器时，需要提供调用端点
                                                       # 该端点由本实验中的 Flask 应用提供
            )]

<a id='2-3'></a>
### 2.3 Properties

在 collection 中，每个数据点的特征字段称为 Properties。

In [ ]:
# 若已存在同名 collection，则先删除
if client.collections.exists("example_collectiom"):
    client.collections.delete("example_collection")
    

In [ ]:
if not client.collections.exists('example_collection'): # 仅当 collection 不存在时创建
    collection = client.collections.create(
            name='example_collection',
            vectorizer_config=vectorizer_config, # 前面定义的向量化配置
            reranker_config=Configure.Reranker.transformers(), # 重排器配置

            properties=[  # 定义 properties
            Property(name="place",vectorize_property_name=True,data_type= DataType.TEXT),
            Property(name="state",vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="description",vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="best_season_to_visit",vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="attractions",vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="budget",vectorize_property_name=True, data_type=DataType.TEXT),
            Property(name="user_ratings", data_type=DataType.NUMBER),
            Property(name="last_updated", data_type=DataType.DATE),

        ]
        )
else:
    collection = client.collections.get("example_collection")

运行后会创建一个 collection 并返回该对象。打印它可以查看 collection 的配置信息。

In [8]:
print(collection)

<weaviate.Collection config={
  "name": "Example_collection",
  "description": null,
  "generative_config": null,
  "inverted_index_config": {
    "bm25": {
      "b": 0.75,
      "k1": 1.2
    },
    "cleanup_interval_seconds": 60,
    "index_null_state": false,
    "index_property_length": false,
    "index_timestamps": false,
    "stopwords": {
      "preset": "en",
      "additions": null,
      "removals": null
    }
  },
  "multi_tenancy_config": {
    "enabled": false,
    "auto_tenant_creation": false,
    "auto_tenant_activation": false
  },
  "object_ttl_config": null,
  "properties": [
    {
      "name": "place",
      "description": null,
      "data_type": "text",
      "index_filterable": true,
      "index_range_filters": false,
      "index_searchable": true,
      "nested_properties": null,
      "tokenization": "word",
      "vectorizer_config": null,
      "vectorizer": null,
      "vectorizer_configs": {
        "text2vec-transformers": {
          "skip": false,
 

如果尝试创建一个已存在的 collection，会抛出异常：

In [ ]:
try:
    collection = client.collections.create(
        name='example_collection',

        vectorizer_config=vectorizer_config, # 前面定义的向量化配置
    
        properties=[  # 定义 properties
        Property(name="place",vectorize_property_name=True,data_type= DataType.TEXT),
        Property(name="state",vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="description",vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="best_season_to_visit",vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="attractions",vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="budget",vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="user_ratings", data_type=DataType.NUMBER),
        Property(name="last_updated", data_type=DataType.DATE),
                 
    ]
    )
except Exception as e:
    print(e)

Collection may not have been created properly.! Unexpected status code: 422, with response body: {'error': [{'message': 'class name Example_collection already exists'}]}.


你也可以获取当前已保存的全部 collections：

In [10]:
client.collections.list_all().keys()

dict_keys(['Example_collection'])

`.list_all()` 的结果是一个字典：键为 collection 名称，值为其配置信息。

<a id='2-4'></a>
### 2.4 向 Collection 添加元素

创建 collection 后，它是空的。接下来需要向其中写入数据。添加元素时后台会发生两个关键步骤：

1. 按 collection 定义进行向量化
2. 更新 HNSW 索引以优化检索（你已在课程中学习）。该过程在后台执行，虽然不可见，但可能需要一定时间

添加元素通常通过 `collection.batch` 完成，它提供了很多实用能力。例如可设置每批发送对象数量、处理导入错误，以及通过减少网络调用提升性能。本示例每次添加一个元素，并仅使用一个并发请求。

你也可以为每个元素添加 uuid（唯一标识），以避免数据库中出现重复条目。

下面看实际示例！

In [ ]:
# 设置固定批大小与并发数的批处理写入
with collection.batch.fixed_size(batch_size=1, concurrent_requests=1) as batch:
    # 遍历数据集（或其子集）
    for document in tqdm(data): # tqdm 用于显示进度条
            # 基于文档内容生成 UUID，确保对象唯一标识
            uuid = generate_uuid5(document)

            # 将对象写入批处理（含 properties 与 UUID）
            # properties 需要是一个字典，键应与定义的 properties 对应
            batch.add_object(
                properties=document,
                uuid=uuid,
            )

100%|██████████| 20/20 [00:01<00:00, 12.16it/s]


很好！现在你已经有一个包含向量的 collection 了！可通过 `len(collection)` 查看向量数量：

In [12]:
len(collection)

20

<a id='3'></a>
## 3 - 在 collection 上查询

在本节中，你将学习如何在 collection 上执行查询。你可以：

- 按元数据查询
- 用语义搜索查询
- 用 BM25 查询
- 使用过滤器查询

下面看一些示例。

<a id='3-1'></a>
### 3.1 过滤器

在深入查询前，先了解 Filters。过滤器用于按条件约束搜索范围，且非常灵活。通常会作为查询参数传入。下面通过示例说明。

In [ ]:
# 获取 2 个对象，并按属性过滤：仅保留 user_ratings >= 3.5 的对象
result = collection.query.fetch_objects(limit = 2, filters = Filter.by_property('user_ratings').greater_or_equal(3.5))

查询结果是一个名为 QueryReturn 的对象：

In [14]:
result

QueryReturn(objects=[Object(uuid=_WeaviateUUIDInt('c99763a3-46a0-59d4-831b-af9bc290260c'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=None, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'description': 'Famous district in Los Angeles known as the entertainment capital of the world.', 'best_season_to_visit': 'Spring', 'attractions': 'Walk of Fame, Hollywood Sign', 'budget': 'Moderate', 'last_updated': datetime.datetime(2023, 10, 1, 0, 0, tzinfo=datetime.timezone.utc), 'state': 'California', 'place': 'Hollywood', 'user_ratings': 4.2}, references=None, vector={}, collection='Example_collection'), Object(uuid=_WeaviateUUIDInt('9e5ba590-8c75-5b53-9b0a-8a9c161004ad'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=None, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'description': 'Bustling pedestrian intersection and major co

你可以通过 `result.objects` 访问其中的对象列表。

In [15]:
result.objects

[Object(uuid=_WeaviateUUIDInt('c99763a3-46a0-59d4-831b-af9bc290260c'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=None, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'description': 'Famous district in Los Angeles known as the entertainment capital of the world.', 'best_season_to_visit': 'Spring', 'attractions': 'Walk of Fame, Hollywood Sign', 'budget': 'Moderate', 'last_updated': datetime.datetime(2023, 10, 1, 0, 0, tzinfo=datetime.timezone.utc), 'state': 'California', 'place': 'Hollywood', 'user_ratings': 4.2}, references=None, vector={}, collection='Example_collection'),
 Object(uuid=_WeaviateUUIDInt('9e5ba590-8c75-5b53-9b0a-8a9c161004ad'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=None, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'description': 'Bustling pedestrian intersection and major commercial hub.', 'be

因此，列表中的每个元素都对应 collection 里的一个对象。

In [16]:
obj = result.objects[0]

你可以查看其 properties，它是一个字典。

In [17]:
obj.properties

{'description': 'Famous district in Los Angeles known as the entertainment capital of the world.',
 'best_season_to_visit': 'Spring',
 'attractions': 'Walk of Fame, Hollywood Sign',
 'budget': 'Moderate',
 'last_updated': datetime.datetime(2023, 10, 1, 0, 0, tzinfo=datetime.timezone.utc),
 'state': 'California',
 'place': 'Hollywood',
 'user_ratings': 4.2}

在本课程中，常用的过滤方式是 `.by_property`。随着后续介绍其他查询方法，你会看到更多过滤示例。

<a id='3-2'></a>
### 3.2 语义搜索

你可以在 collection 上执行语义搜索。该方法基于向量距离计算，返回最接近的对象。你需要传入查询文本，系统会先将其向量化，再与 collection 中元素比较。对应方法是 `.near_text`。

In [18]:
result = collection.query.near_text(query = 'I want suggestions to travel during Winter. I want cheap places.', limit = 4)

In [ ]:
# 遍历结果对象并打印其 properties
for obj in result.objects:
    print_object_properties(obj.properties)

description: Bustling pedestrian intersection and major commercial hub.
best_season_to_visit: Winter
attractions: Broadway Theaters, New Year’s Eve Ball Drop
budget: Low
last_updated: 2023-10-01 00:00:00+00:00
state: New York
place: Times Square
user_ratings: 4.3

description: Park known for its rugged mountains and alpine forests.
best_season_to_visit: Summer
attractions: Going-to-the-Sun Road, Grinnell Glacier
budget: Moderate
last_updated: 2023-10-01 00:00:00+00:00
state: Montana
place: Glacier National Park
user_ratings: 4.8

description: Beautiful park known for its impressive canyons and towering cliffs.
best_season_to_visit: Spring, Fall
attractions: The Narrows, Angels Landing
budget: Moderate
last_updated: 2023-10-01 00:00:00+00:00
state: Utah
place: Zion National Park
user_ratings: 4.7

description: Popular tourist destination known for its beaches and quaint towns.
best_season_to_visit: Summer
attractions: Nantucket Island, Cape Cod National Seashore
budget: Moderate
last_up

You can also already query over the elements with `budget = Low`:

In [20]:
result = collection.query.near_text(query = 'I want suggestions to travel during Winter. I want cheap places.', 
                                    filters = Filter.by_property('budget').equal('Low'),
                                    limit = 4)

In [ ]:
# 遍历结果对象并打印其 properties
for obj in result.objects:
    print_object_properties(obj.properties)

description: Bustling pedestrian intersection and major commercial hub.
best_season_to_visit: Winter
attractions: Broadway Theaters, New Year’s Eve Ball Drop
budget: Low
last_updated: 2023-10-01 00:00:00+00:00
state: New York
place: Times Square
user_ratings: 4.3

description: Famed former prison island located in San Francisco Bay.
best_season_to_visit: Spring, Summer
attractions: Cellhouse Tour, Alcatraz Lighthouse
budget: Low
last_updated: 2023-10-01 00:00:00+00:00
state: California
place: Alcatraz Island
user_ratings: 4.4

description: Historic site of a major Civil War battle.
best_season_to_visit: Spring, Fall
attractions: Gettysburg Museum, Battlefield Tours
budget: Low
last_updated: 2023-10-01 00:00:00+00:00
state: Pennsylvania
place: Gettysburg National Military Park
user_ratings: 4.6

description: Iconic symbol of freedom and democracy in the United States.
best_season_to_visit: Spring, Fall
budget: Low
attractions: Ellis Island, Liberty Island Museum
last_updated: 2023-10-01

You can also pass a list on the possible values on a filter, by using `.contains_any`:

In [22]:
result = collection.query.near_text(query = 'I want suggestions to travel during Winter. I want cheap places.', 
                                    filters = Filter.by_property('budget').contains_any(['Low','Moderate']),
                                    limit = 4)

In [ ]:
# 遍历结果对象并打印其 properties
for obj in result.objects:
    print_object_properties(obj.properties)

description: Bustling pedestrian intersection and major commercial hub.
best_season_to_visit: Winter
attractions: Broadway Theaters, New Year’s Eve Ball Drop
budget: Low
last_updated: 2023-10-01 00:00:00+00:00
state: New York
place: Times Square
user_ratings: 4.3

description: Park known for its rugged mountains and alpine forests.
best_season_to_visit: Summer
attractions: Going-to-the-Sun Road, Grinnell Glacier
budget: Moderate
last_updated: 2023-10-01 00:00:00+00:00
state: Montana
place: Glacier National Park
user_ratings: 4.8

description: Beautiful park known for its impressive canyons and towering cliffs.
best_season_to_visit: Spring, Fall
budget: Moderate
attractions: The Narrows, Angels Landing
last_updated: 2023-10-01 00:00:00+00:00
state: Utah
place: Zion National Park
user_ratings: 4.7

description: Popular tourist destination known for its beaches and quaint towns.
best_season_to_visit: Summer
attractions: Nantucket Island, Cape Cod National Seashore
budget: Moderate
last_up

<a id='3-3'></a>
### 3.3 BM25 搜索

执行 BM25 搜索时，直接调用 `collections.query.bm25` 即可，常见参数 `query`、`limit`、`filters` 都可以传入。

In [24]:
result = collection.query.bm25(query = 'I want suggestions to travel during Winter. I want cheap places.', 
                                    filters = Filter.by_property('budget').contains_any(['Low','Moderate']),
                                    limit = 4)

In [ ]:
# 遍历结果对象并打印其 properties
for obj in result.objects:
    print_object_properties(obj.properties)

description: Bustling pedestrian intersection and major commercial hub.
best_season_to_visit: Winter
budget: Low
attractions: Broadway Theaters, New Year’s Eve Ball Drop
last_updated: 2023-10-01 00:00:00+00:00
state: New York
place: Times Square
user_ratings: 4.3



<a id='3-4'></a>
### 3.4 混合搜索

这种搜索对应课程中介绍的 RRF 搜索。除常规查询参数外，还可以传入 `alpha` 控制混合中 BM25 的权重。

In [26]:
result = collection.query.hybrid(query = 'I want suggestions to travel during Winter. I want cheap places.', 
                                    filters = Filter.by_property('budget').contains_any(['Low','Moderate']),
                                    alpha = 0.3,
                                    limit = 4)

In [ ]:
# 遍历结果对象并打印其 properties
for obj in result.objects:
    print_object_properties(obj.properties)

description: Bustling pedestrian intersection and major commercial hub.
best_season_to_visit: Winter
attractions: Broadway Theaters, New Year’s Eve Ball Drop
budget: Low
last_updated: 2023-10-01 00:00:00+00:00
state: New York
place: Times Square
user_ratings: 4.3

description: Park known for its rugged mountains and alpine forests.
best_season_to_visit: Summer
attractions: Going-to-the-Sun Road, Grinnell Glacier
budget: Moderate
last_updated: 2023-10-01 00:00:00+00:00
state: Montana
place: Glacier National Park
user_ratings: 4.8

description: Beautiful park known for its impressive canyons and towering cliffs.
best_season_to_visit: Spring, Fall
attractions: The Narrows, Angels Landing
budget: Moderate
last_updated: 2023-10-01 00:00:00+00:00
state: Utah
place: Zion National Park
user_ratings: 4.7

description: Popular tourist destination known for its beaches and quaint towns.
best_season_to_visit: Summer
attractions: Nantucket Island, Cape Cod National Seashore
budget: Moderate
last_up

<a id='3-5'></a>
### 3.5 重排（Reranking）

在 Weaviate 中，给查询添加一个参数即可方便地执行重排。下面用语义搜索试试看！

In [ ]:
from weaviate.classes.query import Rerank

response = collection.query.near_text(
    query="'I want suggestions to travel during Winter. I want cheap and fun places.'",  
    limit=5,
    rerank=Rerank(
        prop="attractions",                   # 用于重排的字段
        query="Fun places"  # 若不提供，将使用原始查询
    )
)


In [ ]:
# 遍历结果对象并打印其 properties
for obj in result.objects:
    print_object_properties(obj.properties)

description: Bustling pedestrian intersection and major commercial hub.
best_season_to_visit: Winter
attractions: Broadway Theaters, New Year’s Eve Ball Drop
budget: Low
last_updated: 2023-10-01 00:00:00+00:00
state: New York
place: Times Square
user_ratings: 4.3

description: Park known for its rugged mountains and alpine forests.
best_season_to_visit: Summer
attractions: Going-to-the-Sun Road, Grinnell Glacier
budget: Moderate
last_updated: 2023-10-01 00:00:00+00:00
state: Montana
place: Glacier National Park
user_ratings: 4.8

description: Beautiful park known for its impressive canyons and towering cliffs.
best_season_to_visit: Spring, Fall
attractions: The Narrows, Angels Landing
budget: Moderate
last_updated: 2023-10-01 00:00:00+00:00
state: Utah
place: Zion National Park
user_ratings: 4.7

description: Popular tourist destination known for its beaches and quaint towns.
best_season_to_visit: Summer
attractions: Nantucket Island, Cape Cod National Seashore
budget: Moderate
last_up

In [ ]:
# 别忘了关闭 client！
client.close()

继续加油！你已经完成了本课程中 Weaviate API 的基础部分！